# FitRight: T-Shirt Size Prediction using K-Nearest Neighbors (KNN)

### The Scenario
You are the lead data scientist for "FitRight," a new e-commerce clothing startup. The company wants to reduce return rates by automatically recommending T-shirt sizes (Small or Large) based on a customer's Height (cm) and Weight (kg).


You have a labeled training dataset of 5 past customers:


**Training Dataset:**
| Customer | Height ($x_1$) | Weight ($x_2$) | T-Shirt Size ($y$) |
| :--- | :--- | :--- | :--- |
| A | 158 cm | 58 kg | Small (S) |
| B | 162 cm | 60 kg | Small (S) |
| C | 168 cm | 62 kg | Large (L) |
| D | 170 cm | 64 kg | Large (L) |
| E | 164 cm | 59 kg | Small (S) |

New Customer X:
**  Height: **166 cm**
** Weight: **61 kg**

---

### Knowledge Check
**Question 1:** If you set **$K=3$**, what size will the algorithm recommend for Customer X?  

**A) Small**

B) Large

C) It is a tie

D) Cannot be determined without normalization


In [13]:
import torch 

h = torch.tensor([158,162 ,168 , 170 , 164 ])
w = torch.tensor([58,  60 ,  62 ,  64 ,  59 ])
# l = torch.tensor(['S','S', 'L', 'L', 'S'])
l = torch.tensor([0, 0, 1, 1, 0]) 
x = torch.tensor([166, 61])

print(h  -x[0])
print(w  -x[1])

print("l2 distance")
print((h - x[0])**2 + (w - x[1])**2)

print(torch.argsort((h - x[0])**2 + (w - x[1])**2))

print(l[torch.argsort((h - x[0])**2 + (w - x[1])**2)])

print('l1 distance')
print(torch.abs(h - x[0]) + torch.abs(w - x[1]))
print(torch.argsort(torch.abs(h - x[0]) + torch.abs(w - x[1])))

print(l[torch.argsort(torch.abs(h - x[0]) + torch.abs(w - x[1]))])

tensor([-8, -4,  2,  4, -2])
tensor([-3, -1,  1,  3, -2])
l2 distance
tensor([73, 17,  5, 25,  8])
tensor([2, 4, 1, 3, 0])
tensor([1, 0, 0, 1, 0])
l1 distance
tensor([11,  5,  3,  7,  4])
tensor([2, 4, 1, 3, 0])
tensor([1, 0, 0, 1, 0])


#Put your answer and explanation here...


for K=3 the answer is Small, becuase the 2 of the nearest members of 3 are small sizes that is the reason. 

### Knowledge Check

**Question 2:** If you change the parameter to **$K=1$**, how does the prediction change?  

A) The prediction remains Small; the model is stable.

**B) The prediction changes to Large; the model is sensitive to local noise.**

C) The prediction changes to Large; the model is underfitting.

D) The prediction becomes a tie; the model needs an odd K.


#Put your answer and explanation here...

The prediction changes to Large becuase that is the nearest sample and it is local noise. lets assume the boundary is 167cm but since we do not have enough datapoints in between 166-168 to precisely tell what is what. at boundaries model becomes prone to local noise. 

### Knowledge Check

**Question 3:** If you store Height in meters (1.66m) but Weight in kg (61kg), which feature will dominate the distance?

The result will not change because KNN is scale-invariant.

B) The "Height" feature will dominate the distance calculation, making Weight irrelevant.

**C) The "Weight" feature will dominate the distance calculation, making Height irrelevant.**

D) The algorithm will crash because the units do not match.


#Put your answer and explanation here...

Clearly the weight feature will dominate as the scale matters a lot for the distance calculation like 1.66 - 1.68 is like 0.02 this value is very irrelevant when subtracted with 62- 60 or other scale

### Code Walkthrough

**Question 4:** Output the label for Customer X for each K from 1 to 5. Also, try the code for different combinations of heights and weights.

In [25]:
import pandas as pd
import numpy as np

In [26]:
# 1. Prepare the Training Dataset
data = {
    'Customer': ['A', 'B', 'C', 'D', 'E'],
    'Height': [158, 162, 168, 170, 164],
    'Weight': [58, 60, 62, 64, 59],
    'Size': ['Small', 'Small', 'Large', 'Large', 'Small']
}

In [27]:
df = pd.DataFrame(data)

In [28]:
# 2. Define Customer X
x_height, x_weight = 166, 61
x_height_1 , x_weight_1 = 158, 58
x_height_2 , x_weight_2 = 160, 70
x_height_3 , x_weight_3 = 170, 56
x_height_4 , x_weight_4 = 172, 62
x_height_5 , x_weight_5 = 163, 59

In [29]:
# 3. Calculate Euclidean Distances
# Distance = sqrt((x1_new - x1_old)^2 + (x2_new - x2_old)^2)
df['Distance'] = np.sqrt((df['Height'] - x_height)**2 + (df['Weight'] - x_weight)**2)
df['Distance_1'] = np.sqrt((df['Height'] - x_height_1)**2 + (df['Weight'] - x_weight_1)**2)
df['Distance_2'] = np.sqrt((df['Height'] - x_height_2)**2 + (df['Weight'] - x_weight_2)**2)
df['Distance_3'] = np.sqrt((df['Height'] - x_height_3)**2 + (df['Weight'] - x_weight_3)**2)
df['Distance_4'] = np.sqrt((df['Height'] - x_height_4)**2 + (df['Weight'] - x_weight_4)**2)
df['Distance_5'] = np.sqrt((df['Height'] - x_height_5)**2 + (df['Weight'] - x_weight_5)**2)

In [30]:
df

,Customer,Height,Weight,Size,Distance,Distance_1,Distance_2,Distance_3,Distance_4,Distance_5
0,A,158,58,Small,8.544004,0.000000,12.165525,12.165525,14.560220,5.099020
1,B,162,60,Small,4.123106,4.472136,10.198039,8.944272,10.198039,1.414214
2,C,168,62,Large,2.236068,10.770330,11.313708,6.324555,4.000000,5.830952
3,D,170,64,Large,5.000000,13.416408,11.661904,8.000000,2.828427,8.602325
4,E,164,59,Small,2.828427,6.082763,11.704700,6.708204,8.544004,1.000000


In [31]:
# Sort by distance to find nearest neighbors
df_sorted = df.sort_values(by='Distance').reset_index(drop=True)
df_sorted_1 = df.sort_values(by='Distance_1').reset_index(drop=True)
df_sorted_2 = df.sort_values(by='Distance_2').reset_index(drop=True)
df_sorted_3 = df.sort_values(by='Distance_3').reset_index(drop=True)
df_sorted_4 = df.sort_values(by='Distance_4').reset_index(drop=True)
df_sorted_5 = df.sort_values(by='Distance_5').reset_index(drop=True)

In [32]:
def predict_size(k):
    # Get the top K neighbors
    neighbour_list = []
    preduction_list = []
    neighbour_list.append(df_sorted.head(k))
    preduction_list.append(neighbour_list[0]['Size'].mode()[0])

    neighbour_list.append(df_sorted_1.head(k))
    preduction_list.append(neighbour_list[1]['Size'].mode()[0])

    neighbour_list.append(df_sorted_2.head(k))
    preduction_list.append(neighbour_list[2]['Size'].mode()[0])
    neighbour_list.append(df_sorted_3.head(k))
    preduction_list.append(neighbour_list[3]['Size'].mode()[0])
    neighbour_list.append(df_sorted_4.head(k))
    preduction_list.append(neighbour_list[4]['Size'].mode()[0])
    neighbour_list.append(df_sorted_5.head(k))
    preduction_list.append(neighbour_list[5]['Size'].mode()[0])
    return preduction_list, neighbour_list

In [37]:
# --- STUDENT INTERACTIVE SECTION ---
# Change the value of K here (1 to 5)



for i in range(1,6):
    prediction, nearest_neighbors = predict_size(i)
    print(f"Results for K = {i}")
    print("-" * 30)
    for j in range(1,7):
        print(f'for customer : {j} ')
        print(f"Prediction for Customer X: {prediction[j-1]}")
        print("\nNearest Neighbors used:")
        print(nearest_neighbors[j-1][['Customer', 'Height', 'Weight', 'Distance', 'Size']])
        print("\n")

Results for K = 1
------------------------------
for customer : 1 
Prediction for Customer X: Large

Nearest Neighbors used:
  Customer  Height  Weight  Distance   Size
0        C     168      62  2.236068  Large


for customer : 2 
Prediction for Customer X: Small

Nearest Neighbors used:
  Customer  Height  Weight  Distance   Size
0        A     158      58  8.544004  Small


for customer : 3 
Prediction for Customer X: Small

Nearest Neighbors used:
  Customer  Height  Weight  Distance   Size
0        B     162      60  4.123106  Small


for customer : 4 
Prediction for Customer X: Large

Nearest Neighbors used:
  Customer  Height  Weight  Distance   Size
0        C     168      62  2.236068  Large


for customer : 5 
Prediction for Customer X: Large

Nearest Neighbors used:
  Customer  Height  Weight  Distance   Size
0        D     170      64       5.0  Large


for customer : 6 
Prediction for Customer X: Small

Nearest Neighbors used:
  Customer  Height  Weight  Distance   Size
0

In [ ]:
K_VALUE = 3 
prediction, nearest_neighbors = predict_size(K_VALUE)


In [34]:
print(f"Results for K = {K_VALUE}")
print("-" * 30)
print(f"Prediction for Customer X: {prediction[0]}")
print("\nNearest Neighbors used:")
print(nearest_neighbors[0][['Customer', 'Height', 'Weight', 'Distance', 'Size']])

Results for K = 3
------------------------------
Prediction for Customer X: Small

Nearest Neighbors used:
  Customer  Height  Weight  Distance   Size
0        C     168      62  2.236068  Large
1        E     164      59  2.828427  Small
2        B     162      60  4.123106  Small


In [35]:
print(f"Results for K = {K_VALUE}")
print("-" * 30)
print(f"Prediction for Customer X: {prediction[1]}")
print("\nNearest Neighbors used:")
print(nearest_neighbors[1][['Customer', 'Height', 'Weight', 'Distance', 'Size']])

Results for K = 3
------------------------------
Prediction for Customer X: Small

Nearest Neighbors used:
  Customer  Height  Weight  Distance   Size
0        A     158      58  8.544004  Small
1        B     162      60  4.123106  Small
2        E     164      59  2.828427  Small


In [36]:
print(f"Results for K = {K_VALUE}")
print("-" * 30)
print(f"Prediction for Customer X: {prediction[2]}")
print("\nNearest Neighbors used:")
print(nearest_neighbors[2][['Customer', 'Height', 'Weight', 'Distance', 'Size']])

Results for K = 3
------------------------------
Prediction for Customer X: Large

Nearest Neighbors used:
  Customer  Height  Weight  Distance   Size
0        B     162      60  4.123106  Small
1        C     168      62  2.236068  Large
2        D     170      64  5.000000  Large
